In [1]:
# Core RAG Frameworks
!pip install langchain
!pip install langchain-openai
!pip install langchain-community
!pip install langchain-text-splitters
!pip install langchain-experimental
!pip install langgraph


In [2]:
import json
import os

from openai import OpenAI
from google.colab import userdata

from langchain_core.documents import Document
from langchain_experimental.text_splitter import SemanticChunker  # AI-powered semantic chunking
from langchain_community.vectorstores import Chroma  # Vector database
from langchain_openai import OpenAIEmbeddings  # OpenAI embedding function
from langchain_core.documents import Document  # Document abstraction


from langchain_text_splitters import (  # Various splitting strategies
    RecursiveCharacterTextSplitter,  # Best general-purpose splitter
    CharacterTextSplitter,  # Simple split by character count
    MarkdownHeaderTextSplitter,  # Splits based on markdown headers
    HTMLHeaderTextSplitter  # Splits based on HTML tags
)


/tmp/ipykernel_7374/929496560.py:8: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker  # AI-powered semantic chunking


In [3]:
# =============================================================================
# LOAD DATA: Support Tickets
# =============================================================================
print("\nLoading support tickets...")
with open('./sample_data/synthetic_tickets.json', 'r') as f:
    tickets = json.load(f)
print(f"Loaded {len(tickets)} support tickets")


Loading support tickets...
Loaded 20 support tickets


In [4]:
# Display sample ticket - understand what we're working with
print("\n" + "="*80)
print("SAMPLE TICKET:")
print("="*80)
sample = tickets[0]
print(f"ID: {sample['ticket_id']}")
print(f"Title: {sample['title']}")
print(f"Description: {sample['description'][:200]}...")
print(f"Category: {sample['category']}")
print(f"Priority: {sample['priority']}")

print("\n" + "="*80)
print("PART 1: Chunking Strategies")
print("="*80)


SAMPLE TICKET:
ID: TICK-001
Title: Users unable to log in after password reset
Description: Multiple users reporting authentication failures after performing password reset. Error message: 'Invalid credentials'. Issue started after recent security patch deployment....
Category: Authentication
Priority: High

PART 1: Chunking Strategies


In [5]:
# -----------------------------------------------------------------------------
# First, convert our ticket data into LangChain Document objects
# Document = page_content (text) + metadata (structured info for filtering)
# -----------------------------------------------------------------------------

documents = []
for ticket in tickets:
    # Combine all ticket fields into a single text block
    # TIP: Include all relevant context that helps understand the document
    full_text = f"""
Ticket ID: {ticket['ticket_id']}
Title: {ticket['title']}
Category: {ticket['category']}
Priority: {ticket['priority']}
Description: {ticket['description']}
Resolution: {ticket['resolution']}
    """.strip()


    # Create Document object with metadata
    # Metadata is CRUCIAL - it enables filtering later!
    # Example: "Find similar tickets, but only in the 'Authentication' category"
    doc = Document(
        page_content=full_text,  # The actual text content
        metadata={
            'ticket_id': ticket['ticket_id'],   # For identifying results
            'category': ticket['category'],      # For category filtering
            'description': ticket['description']       # For priority filtering
        }
    )
    documents.append(doc)

print(f"Created {len(documents)} documents")
print(f"\nSample document length: {len(documents[0].page_content)} characters")

Created 20 documents

Sample document length: 523 characters


In [6]:
# Vector Stores
!pip install chromadb
!pip install langchain-chroma

In [7]:
# ============================================================================
# PART 2: Chroma Vector Store
# ============================================================================

print("\n" + "="*80)
print("PART 2: Chroma Vector Store")
print("="*80)

# Use LangChain's embedding wrapper (handles API calls internally)
embeddings_model = OpenAIEmbeddings(
    model='text-embedding-3-small',
    api_key=userdata.get('OPENAI_APIKEY')
)



print("\nBuilding Chroma vector store...")

# Clean up any existing collection to avoid duplicates on re-run
# from_documents() APPENDS to existing collections, so without this
# you'd get duplicate documents (and duplicate search results!) each run
existing_store = Chroma(
    collection_name="support_tickets",
    persist_directory="./chroma_db"
)
existing_store.delete_collection()

# from_documents() handles everything:
#   1. Extracts text from each Document
#   2. Generates embeddings via the embedding model
#   3. Stores vectors + metadata + original text
#   4. Persists to disk (if persist_directory specified)
chroma_store = Chroma.from_documents(
    documents=documents,              # Our LangChain Document objects
    embedding=embeddings_model,       # OpenAI embeddings
    collection_name="support_tickets",# Like a "table" in a database
    persist_directory="./chroma_db"   # Save to disk for persistence
)
print("✓ Chroma store created and persisted")


PART 2: Chroma Vector Store

Building Chroma vector store...


/tmp/ipykernel_7374/3778981295.py:22: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  existing_store = Chroma(


✓ Chroma store created and persisted


In [8]:
# -----------------------------------------------------------------------------
# Basic Similarity Search with query1
# -----------------------------------------------------------------------------

query1 = "Authentication problems after password reset"

print(f"\nSearching in Chroma: '{query1}'")
chroma_results = chroma_store.similarity_search(query1, k=3)

print(f"\nTop {len(chroma_results)} results:")
for i, doc in enumerate(chroma_results, 1):
    print(f"\n#{i}")
    print(f"Ticket: {doc.metadata['ticket_id']}")
    print(f"Category: {doc.metadata['category']}")
    print(f"Description: {doc.metadata['description']}")


Searching in Chroma: 'Authentication problems after password reset'

Top 3 results:

#1
Ticket: TICK-001
Category: Authentication
Description: Multiple users reporting authentication failures after performing password reset. Error message: 'Invalid credentials'. Issue started after recent security patch deployment.

#2
Ticket: TICK-016
Category: Authentication
Description: Users unable to log in with TOTP codes from authenticator apps. Codes shown as invalid even when entered immediately after generation. Affects random subset of users.

#3
Ticket: TICK-011
Category: Authentication
Description: Single Sign-On integration with corporate SAML provider failing. Users redirected to SSO portal but then receive 'Invalid SAML response' error when redirected back. Started after v2.5 upgrade.


In [9]:
# -----------------------------------------------------------------------------
# Basic Similarity Search with query2
# -----------------------------------------------------------------------------

query2 = "Database is timing out frequently"

print(f"\nSearching in Chroma: '{query2}'")
chroma_results = chroma_store.similarity_search(query2, k=3)

print(f"\nTop {len(chroma_results)} results:")
for i, doc in enumerate(chroma_results, 1):
    print(f"\n#{i}")
    print(f"Ticket: {doc.metadata['ticket_id']}")
    print(f"Category: {doc.metadata['category']}")
    print(f"Description: {doc.metadata['description']}")


Searching in Chroma: 'Database is timing out frequently'

Top 3 results:

#1
Ticket: TICK-002
Category: Database
Description: Application experiencing intermittent 500 errors. Logs show 'connection pool exhausted' and 'timeout waiting for connection'. Affects approximately 20% of requests during peak hours.

#2
Ticket: TICK-014
Category: Authentication
Description: Users complaining about being logged out too frequently. Currently set to 15 minutes. Particularly problematic for users filling out long forms, losing unsaved data when session expires.

#3
Ticket: TICK-010
Category: Performance
Description: Admin dashboard takes 30+ seconds to load. Browser DevTools show multiple slow database queries. Page becomes unresponsive during load. Affects all users with admin privileges.
